In [48]:
from langgraph.graph import StateGraph,START,END
from dotenv import load_dotenv
from typing import TypedDict
from langchain_groq import ChatGroq

In [49]:
load_dotenv()

model = ChatGroq(
    model_name="llama-3.1-8b-instant",
    temperature=0.3,
)

In [50]:
class blogstate(TypedDict):
    title :str
    outline :str
    blog : str
    evaluation_rating :str

In [51]:
graph=StateGraph(blogstate)

In [52]:
def create_outline(state : blogstate) -> blogstate:
    title=state['title']

    prompt=f"generate and outline for a blog on the topic {title}"

    outline=model.invoke(prompt).content

    state['outline']=outline

    return state

In [53]:
def create_blog(state :blogstate) -> blogstate:

    title=state['title']
    outline=state['outline']

    prompt=f"generate a blog for the topic {title} having the outline {outline}"

    blog=model.invoke(prompt).content

    state['blog']=blog

    return state

In [54]:
def evaluate(state:blogstate) ->blogstate:
    outline=state['outline']
    blog=state['blog']

    prompt=f"based on the outline {outline} , judge my blog {blog}"
    
    eval=model.invoke(prompt).content

    state['evaluation_rating']=eval

    return state

In [55]:
graph.add_node('outline',create_outline)
graph.add_node('blog',create_blog)
graph.add_node('evaluate',evaluate)

In [56]:
graph.add_edge(START,'outline')
graph.add_edge('outline','blog')
graph.add_edge('blog','evaluate')
graph.add_edge('evaluate',END)

In [57]:
workflow=graph.compile()

In [58]:
initial_state={'title':'iron man the best'}
final_state=workflow.invoke(initial_state)
print(final_state)
print(final_state['outline'])
print(final_state['blog'])
print(final_state['evaluation_rating'])

{'title': 'iron man the best', 'outline': 'Here\'s a suggested outline for a blog on "Iron Man: The Best":\n\n**I. Introduction**\n\n* Briefly introduce the topic of the blog: Iron Man as the best superhero\n* Mention the reasons why Iron Man stands out among other superheroes\n* Thesis statement: Iron Man is the best superhero due to his intelligence, technological advancements, and relatable character.\n\n**II. Intelligence and Strategic Thinking**\n\n* Discuss Tony Stark\'s intelligence and strategic thinking as key factors in his success as Iron Man\n* Provide examples from the Marvel Cinematic Universe (MCU) where Iron Man\'s intelligence saves the day\n* Explain how his intelligence and strategic thinking make him a more effective superhero than others\n\n**III. Technological Advancements**\n\n* Discuss the advanced technology developed by Tony Stark as Iron Man, such as the suit\'s AI, repulsor technology, and advanced sensors\n* Explain how these technological advancements give